# Day 44: pandas DataFrames

**Today's question:** three months of expenses are in a CSV file. *Where did the money go?*

NumPy arrays hold one type and have no column names. Real tables mix text and numbers and have named columns.
**pandas** gives us exactly that: a **DataFrame**. Before running a cell marked 🤔, predict what it will show.

## 1. A DataFrame is a table with named columns

By convention pandas is imported as `pd`. A small DataFrame can be built from a dict: key = column name,
value = the column's values.

In [ ]:
import pandas as pd

shopping = pd.DataFrame({
    "item": ["rice", "milk", "soap", "apples"],
    "category": ["food", "food", "home", "food"],
    "price": [4.50, 1.20, 2.00, 3.10],
})
shopping

The numbers on the left (0–3) are the **index**: a label for each row. One column on its own is a **Series**:
like a NumPy array, plus a name and an index.

In [ ]:
prices = shopping["price"]
print(type(prices))
print(prices.mean(), prices.max())

## 2. Loading and inspecting a CSV

In [ ]:
from pathlib import Path

DATA_DIR = Path("../../../datasets")
expenses = pd.read_csv(DATA_DIR / "expenses.csv")
expenses.head()

🤔 **Predict:** the file has 149 lines including the header. What will `.shape` show?

In [ ]:
expenses.shape

`.info()` lists every column, how many values are present (non-null) and its type. No `float()` needed: pandas
detected that `amount` is a number.

In [ ]:
expenses.info()

`.describe()` gives summary statistics of the numeric columns in one go.

In [ ]:
expenses.describe().round(2)

🤔 **Discuss:** the mean is 39.94 but the median (`50%`) is only 26.16. Why? (Remember Ken's 0 yesterday.)

## 3. Selecting columns

In [ ]:
print("Total spent:", round(expenses["amount"].sum(), 2))

Several columns: put a **list** of names inside the brackets, hence the double brackets.

In [ ]:
expenses[["date", "description", "amount"]].head(3)

In [ ]:
# Mistake: column names are case-sensitive
expenses["Amount"]

## 4. Filtering rows with conditions

Exactly the mask idea from Day 43: a comparison gives `True`/`False` per row, and the DataFrame keeps the `True` rows.

In [ ]:
big = expenses["amount"] > 50
print(big.head())
expenses[big]

🤔 **Predict:** how many **food** purchases were paid in **cash**? Same rules as NumPy: `&` / `|` and brackets
around every condition.

In [ ]:
food_cash = expenses[(expenses["category"] == "food") & (expenses["payment_method"] == "cash")]
print(len(food_cash), "purchases, total", round(food_cash["amount"].sum(), 2))

In [ ]:
fun = expenses[(expenses["category"] == "entertainment") | (expenses["category"] == "shopping")]
fun.head()

In [ ]:
# Mistake: `and` cannot combine two whole columns
expenses[(expenses["category"] == "food") and (expenses["amount"] > 30)]

## 5. Sorting

In [ ]:
expenses.sort_values("amount", ascending=False).head(5)

🤔 **Predict:** what will be at the top if we leave out `ascending=False`?

Combine steps: the 5 biggest purchases that are *not* rent.

In [ ]:
not_rent = expenses[expenses["category"] != "rent"]
not_rent.sort_values("amount", ascending=False).head(5)

## 6. New columns

Assign to a new column name. The calculation runs on every row (vectorised, like NumPy).

In [ ]:
expenses["is_big"] = expenses["amount"] >= 50
expenses.head(3)

Text columns have a `.str` helper that applies string methods and slicing to every row. The first 7 characters of
`"2026-09-01"` are the month `"2026-09"`. (We use more `.str` methods on Day 45.)

In [ ]:
expenses["month"] = expenses["date"].str[:7]
expenses[["date", "month", "amount"]].head(3)

## 7. Counting categories: `value_counts`

🤔 **Predict:** which category appears most often?

In [ ]:
expenses["category"].value_counts()

In [ ]:
expenses["payment_method"].value_counts()

## 8. Group and summarise: `groupby`

**Analogy:** sort the receipts into piles by category, then add up each pile.

`df.groupby("pile column")["value column"].sum()`

In [ ]:
expenses.groupby("category")["amount"].sum().sort_values(ascending=False)

In [ ]:
expenses.groupby("category")["amount"].mean().round(2)

🤔 **Predict:** food has the most transactions. Does it also have the biggest **total**?

By month, using the column we created in section 6:

In [ ]:
expenses.groupby("month")["amount"].sum()

## 9. Same tools, another dataset

The students file from Days 41–43, now in one line.

In [ ]:
students = pd.read_csv(DATA_DIR / "students.csv")
print(students.shape)
students.groupby("program")["final_score"].mean().round(1)

`passed` is 0/1, so its **mean is the pass rate** (same trick as `mask.mean()` yesterday).

In [ ]:
students.groupby("has_job")["passed"].mean().round(3)

In [ ]:
keen = students[(students["study_hours"] >= 10) & (students["attendance_pct"] >= 90)]
print(len(keen), "keen students, pass rate", round(keen["passed"].mean(), 3))

Remember Day 41? It took us a whole function and a loop to get the pass rate per group. Now it is one line.

## Recap

| Task | pandas |
|------|--------|
| load / look | `pd.read_csv(path)`, `.head()`, `.shape`, `.info()`, `.describe()` |
| one column / several | `df["col"]` (Series) / `df[["a", "b"]]` (DataFrame) |
| filter rows | `df[(df["a"] > 1) & (df["b"] == "x")]` |
| sort | `df.sort_values("col", ascending=False)` |
| new column | `df["new"] = df["a"] * 2` |
| count categories | `df["col"].value_counts()` |
| summarise per group | `df.groupby("group")["value"].sum()` / `.mean()` |

**Next time:** the messy `students_raw.csv` again. Cleaning it with pandas, then telling its story with charts.